# Simulating Stock Prices with Geometric Brownian Motion

Stock prices are uncertain, so quantitative analysts *simulate* thousands of possible futures and study the results. The most famous model is **Geometric Brownian Motion (GBM)**, used in options pricing, risk management and portfolio simulation. In this assignment you will build a GBM simulator using only Python basics from CSCA08: functions, loops, lists and the `math` / `random` modules.

### How to complete this notebook
- Replace every `___` (three underscores) with your own code. A cell will not run until all blanks in it are filled.
- Exercises are sorted by difficulty: ⭐ Easy → ⭐⭐ Medium → ⭐⭐⭐ Harder (including the vectorized NumPy section) → ⭐⭐⭐⭐ Challenge.
- **You do not need to understand stochastic calculus :)** The maths section below is read-only; every exercise only needs the step formula and CSCA08 Python.
- Run the **Check** cell after each exercise. Don't move on until it passes.
- Do not change function names, parameters, or the docstrings!

## The Model in Plain English
A stock price starts at `S0`. Each small time step of length `dt` (in years), the price is multiplied by a random factor:

$$S_{t+dt} = S_t \cdot \exp\Big(\underbrace{(\mu - \tfrac{1}{2}\sigma^2)\,dt}_{\text{drift}} + \underbrace{\sigma\sqrt{dt}\,Z}_{\text{shock}}\Big)$$

| Symbol | Meaning | Example |
|---|---|---|
| `S0` | starting price | 100 |
| `mu` (μ) | expected yearly return | 0.10 (10%) |
| `sigma` (σ) | yearly volatility (how wild the price moves) | 0.30 (30%) |
| `dt` | length of one step in years, `T / n_steps` | 1/100 of a year |
| `Z` | a random number from the standard normal distribution | `random.gauss(0, 1)` |

**Drift** is the predictable part of the move; **shock** is the random part. Note that the price is *multiplied* by a positive number (`exp` is always > 0), so a simulated price can never go negative.

---
## 📖 Where the Formula Comes From (read-only)
The video starts from a *stochastic differential equation* (SDE). **You do not need to follow the calculus.** This section translates the video into the code you will write.

$$dS_t = \mu S_t\,dt + \sigma S_t\,dW_t$$

**In plain English:** in a tiny moment `dt`, the price changes by a *predictable part* (`mu * S * dt`) plus a *random part* (`sigma * S * dW`). Here `W_t` is **Brownian motion**, a random walk whose change over a step of length `dt` is normally distributed with mean 0 and standard deviation `sqrt(dt)`. That is exactly the `np.random.normal(0, np.sqrt(dt))` in the video. It is the same thing as `sqrt(dt) * Z` where `Z` is a standard normal number.

Solving the SDE (using a tool called Itô's lemma) gives the **explicit solution**:

$$\log S_t = \log S_0 + \Big(\mu - \frac{\sigma^2}{2}\Big)t + \sigma W_t \qquad\Longleftrightarrow\qquad S_t = S_0\, e^{(\mu - \sigma^2/2)t + \sigma W_t}$$

Two consequences you will use:
1. **The log of the price is normally distributed**, so the price itself is *log-normal* and can never be negative.
2. Applying the formula over one small step of length `dt` gives exactly the step formula we use below. The `- sigma**2 / 2` is the **Itô correction**; you can accept it as given.


### Glossary
- **SDE:** an equation describing how something changes with both a predictable and a random part.
- **Brownian motion `W_t`:** a continuous random walk.
- **Itô correction:** the `-sigma**2 / 2` adjustment that keeps the expected price right.
- **Log-normal:** a value whose logarithm is normally distributed.
- **Monte Carlo:** estimating an answer by running many random simulations.

In [ ]:
import math
import random
from typing import List, Tuple

import matplotlib.pyplot as plt

# Parameters we will reuse (same values as the video)
S0 = 100.0     # initial stock price
MU = 0.10      # drift coefficient (yearly)
SIGMA = 0.30   # volatility (yearly)
T = 1.0        # time in years
N_STEPS = 100  # number of steps (n in the video)
M = 100        # number of simulated paths

---
## ⭐ Level 1 – Fill in the blanks: one formula at a time

### Exercise 1.1: drift and shock
Fill in the two blanks in each function using the formulas above.

In [ ]:
def drift_term(mu: float, sigma: float, dt: float) -> float:
    """Return the drift part of one GBM step: (mu - 0.5 * sigma**2) * dt.

    >>> round(drift_term(0.08, 0.20, 1 / 252), 6)
    0.000238
    """
    return (mu - ___ * sigma ** 2) * ___


def shock_term(sigma: float, dt: float, z: float) -> float:
    """Return the random part of one GBM step: sigma * sqrt(dt) * z.

    >>> shock_term(0.20, 1 / 252, 0.0)
    0.0
    >>> round(shock_term(0.20, 0.25, 1.0), 6)
    0.1
    """
    return sigma * math.___(dt) * ___

In [ ]:
# Check 1.1
assert abs(drift_term(0.08, 0.20, 1 / 252) - 0.06 / 252) < 1e-12
assert shock_term(0.20, 1 / 252, 0.0) == 0.0
assert abs(shock_term(0.20, 0.25, 1.0) - 0.1) < 1e-12
assert abs(shock_term(0.20, 0.25, -2.0) + 0.2) < 1e-12
print("Exercise 1.1 passed ✅")

### Exercise 1.2: one step of the price
Combine the two terms: `new_price = price * exp(drift + shock)`.

In [ ]:
def gbm_step(price: float, mu: float, sigma: float, dt: float, z: float) -> float:
    """Return the price after one GBM time step, given the random number z.

    >>> round(gbm_step(100.0, 0.08, 0.20, 1 / 252, 0.0), 4)
    100.0238
    """
    drift = ___(mu, sigma, dt)
    shock = ___(sigma, dt, z)
    return price * math.___(drift + shock)

In [ ]:
# Check 1.2
assert abs(gbm_step(100.0, 0.08, 0.20, 1 / 252, 0.0) - 100 * math.exp(0.06 / 252)) < 1e-9
assert gbm_step(100.0, 0.08, 0.20, 1 / 252, -50.0) > 0   # prices never go negative
print("Exercise 1.2 passed ✅")

---
## ⭐⭐ Level 2 – Fill in the blanks: build a full path with a loop

### Exercise 2.1: simulate one price path
Read the code carefully, then fill in the blanks. A path with `n_steps` steps has `n_steps + 1` prices (the starting price plus one per step).

In [ ]:
def simulate_path(s0: float, mu: float, sigma: float, t: float, n_steps: int) -> List[float]:
    """Return one simulated GBM price path as a list of n_steps + 1 prices.
    The first price is s0. Uses random.gauss(0, 1) for each step.

    >>> random.seed(1)
    >>> path = simulate_path(100.0, 0.08, 0.20, 1.0, 10)
    >>> len(path)
    11
    >>> path[0]
    100.0
    """
    dt = ___ / n_steps
    path = [s0]
    for _ in range(___):
        z = random.gauss(0, ___)
        new_price = gbm_step(path[-1], mu, sigma, dt, z)
        path.___(new_price)
    return path

In [ ]:
# Check 2.1
random.seed(1)
p = simulate_path(100.0, 0.08, 0.20, 1.0, 252)
assert len(p) == 253
assert p[0] == 100.0
assert all(x > 0 for x in p)
assert p != [100.0] * 253   # the price should actually move!
print("Exercise 2.1 passed ✅")

### Exercise 2.2: the second approach: add log-returns instead of multiplying
The video mentions two ways to build a path. In Exercise 2.1 you *multiplied* the price by a factor each step. The alternative works with the **log of the price**: add the drift and the shock to the log price at every step, then convert back with `exp`. With the same random numbers, both methods give the same path.

In [ ]:
def simulate_path_log(s0: float, mu: float, sigma: float, t: float, n_steps: int) -> List[float]:
    """Return one GBM path (n_steps + 1 prices) by accumulating the LOG price.
    Uses the same random numbers, in the same order, as simulate_path.

    >>> random.seed(1)
    >>> a = simulate_path(100.0, 0.10, 0.30, 1.0, 5)
    >>> random.seed(1)
    >>> b = simulate_path_log(100.0, 0.10, 0.30, 1.0, 5)
    >>> all(abs(x - y) < 1e-9 for x, y in zip(a, b))
    True
    """
    dt = t / n_steps
    log_price = math.log(s0)
    path = [s0]
    for _ in range(n_steps):
        z = random.gauss(0, 1)
        log_price = log_price + ___(mu, sigma, dt) + ___(sigma, dt, z)
        path.append(math.___(log_price))
    return path

In [ ]:
# Check 2.2: both approaches must agree when they use the same random numbers
random.seed(5)
a = simulate_path(100.0, 0.10, 0.30, 1.0, 100)
random.seed(5)
b = simulate_path_log(100.0, 0.10, 0.30, 1.0, 100)
assert len(b) == 101 and b[0] == 100.0
assert all(abs(x - y) < 1e-6 for x, y in zip(a, b))
print("Exercise 2.2 passed ✅")

### Exercise 2.3: a time axis in years
The video plots against time in **years**, not step numbers. Write a list of `n_steps + 1` evenly spaced times from `0` to `t`.

In [ ]:
def make_times(t: float, n_steps: int) -> List[float]:
    """Return n_steps + 1 evenly spaced times in years from 0.0 to t.

    >>> make_times(1.0, 4)
    [0.0, 0.25, 0.5, 0.75, 1.0]
    """
    dt = ___ / n_steps
    return [i * ___ for i in range(___)]

In [ ]:
# Check 2.3
ts = make_times(1.0, 4)
assert ts == [0.0, 0.25, 0.5, 0.75, 1.0]
assert len(make_times(2.0, 100)) == 101
assert abs(make_times(2.0, 100)[-1] - 2.0) < 1e-9
print("Exercise 2.3 passed ✅")

### Exercise 2.4: plot your path
Fill in the blanks to plot one path against time in years, with labelled axes and a title showing the parameters. Run it several times with different seeds. Then try `SIGMA = 0.05` and `SIGMA = 0.60` in the parameter cell. What happens to the shape of the path?

In [ ]:
times = make_times(T, N_STEPS)
random.seed(8)
path = simulate_path(S0, MU, SIGMA, T, N_STEPS)

plt.figure(figsize=(8, 4))
plt.plot(___, ___)
plt.xlabel("Years (t)")
plt.ylabel(___)
plt.title(f"One GBM path: S0={___}, mu={___}, sigma={___}")
plt.show()

---
## ⭐⭐⭐ Level 3 – Complete the function body

This time there are **no blanks**. You get the docstring and write the whole body yourself.

### Exercise 3.1: many paths
Use `simulate_path` inside a loop to build a list of `n_paths` paths (a list of lists).

In [ ]:
def simulate_paths(s0: float, mu: float, sigma: float, t: float,
                   n_steps: int, n_paths: int) -> List[List[float]]:
    """Return a list of n_paths independent GBM price paths.
    Each path is a list of n_steps + 1 prices, as returned by simulate_path.

    >>> random.seed(2)
    >>> paths = simulate_paths(100.0, 0.08, 0.20, 1.0, 5, 3)
    >>> len(paths)
    3
    >>> len(paths[0])
    6
    """
    # TODO: write the function body
    pass

In [ ]:
# Check 3.1
random.seed(2)
ps = simulate_paths(100.0, 0.08, 0.20, 1.0, 5, 3)
assert len(ps) == 3
assert all(len(p) == 6 for p in ps)
assert all(p[0] == 100.0 for p in ps)
assert ps[0] != ps[1]   # paths must be independent, not copies
print("Exercise 3.1 passed ✅")

### Exercise 3.2: summarize the final prices
Write both functions from scratch. Built-ins like `sum` and `len` are fine, but no `statistics` or `numpy` imports.

In [ ]:
def final_prices(paths: List[List[float]]) -> List[float]:
    """Return the last price of each path, in the same order as paths.

    >>> final_prices([[1.0, 2.0, 3.0], [1.0, 5.0, 4.0]])
    [3.0, 4.0]
    """
    # TODO: write the function body
    pass


def average(values: List[float]) -> float:
    """Return the mean of a non-empty list of numbers.

    >>> average([1.0, 2.0, 3.0, 6.0])
    3.0
    """
    # TODO: write the function body
    pass

In [ ]:
# Check 3.2
assert final_prices([[1.0, 2.0, 3.0], [1.0, 5.0, 4.0]]) == [3.0, 4.0]
assert final_prices([]) == []
assert average([1.0, 2.0, 3.0, 6.0]) == 3.0
print("Exercise 3.2 passed ✅")

### Exercise 3.3: does the simulator behave correctly?
GBM has a known answer: the **expected final price** is `S0 * exp(mu * T)`. With our parameters that is about **110.52**. Run 2000 paths and compare. (This cell is complete. Read it, run it, and answer the questions in the next cell.)

In [ ]:
random.seed(42)
paths = simulate_paths(S0, MU, SIGMA, T, N_STEPS, 2000)
finals = final_prices(paths)
theory = S0 * math.exp(MU * T)
print(f"Simulated average final price: {average(finals):.2f}")
print(f"Theoretical expected price:    {theory:.2f}")
assert abs(average(finals) - theory) < 4
print("Exercise 3.3 passed ✅")

plt.figure(figsize=(8, 4))
times = make_times(T, N_STEPS)
for p in paths[:50]:
    plt.plot(times, p, linewidth=0.7, alpha=0.6)
plt.title(f"50 simulated GBM paths: S0={S0}, mu={MU}, sigma={SIGMA}")
plt.xlabel("Years (t)")
plt.ylabel("Stock Price (S_t)")
plt.show()

**Short answer questions** (double-click to edit):

1. Why doesn't the simulated average match 110.52 *exactly*? What could you change to get closer?
   *Your answer:*
2. If `sigma` is doubled, what happens to the spread of the final prices? What happens to their average?
   *Your answer:*
3. Why do we subtract `0.5 * sigma**2` in the drift?
   *Your answer:*

---
## ⭐⭐⭐ Level 4 – Simulating GBM: Vectorized

So far each path was built **one step at a time** in a loop: the price at step `i` depends on the price at step `i - 1`. There is a faster and more elegant way: generate **all the randomness at once** in one big matrix, then turn it into prices with a few whole-matrix operations. There are no loops over time steps and no loops over paths.

### The idea
Let `N = n_paths` and `K = n_steps` (so `K = T / dt`).

1. **Random matrix.** Draw `Z`, an `N × K` matrix of standard normal numbers. Row `i` is path `i`; column `j` is time step `j`.
2. **Log-increments.** Compute `X = drift + shock` for every entry at once, using the same terms as Level 1:
   `X[i][j] = (mu - 0.5*sigma**2)*dt + sigma*sqrt(dt)*Z[i][j]`
3. **Running totals.** The log of the price after step `k` is `log(S0)` plus the sum of the first `k` entries of the row. A running total is a **matrix multiplication**: multiply `X` by the `K × K` upper-triangular matrix `U` of ones, where `U[j][k] = 1` if `j <= k` and `0` otherwise.
4. **Back to prices.** `S = S0 * exp(running totals)`, with a column of zeros added at the front for the starting price (`exp(0) = 1`).

$$\underbrace{X}_{N \times K}\;\underbrace{U}_{K \times K} = \text{running totals of each row of } X \qquad\Longrightarrow\qquad S = S_0\, e^{\,\text{running totals}}$$

> **Link to the video:** the video builds a *time × paths* matrix of price **factors** `exp(drift + shock)` and uses `cumprod`. Since `exp(a) * exp(b) = exp(a + b)`, multiplying factors and adding log-increments give the same prices. Here rows are paths (like our list of paths from Level 3), whereas the video's matrix is transposed (`.T`).

### NumPy mini-primer
CSCA08 hasn't covered NumPy yet, so here are the only five things you need. Run this cell and read the output.

In [ ]:
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])          # a 2 x 3 array: 2 rows, 3 columns
print(a.shape)                     # (2, 3)
print(a * 10)                      # operations apply to EVERY entry at once
print(a.sum(axis=1))               # axis=1: combine across columns, one answer per row -> [ 6 15]

U = np.triu(np.ones((3, 3)))       # upper-triangular matrix of ones
print(U)
print(a @ U)                       # '@' is matrix multiplication: each row becomes its running total
print(np.cumsum(a, axis=1))        # the built-in way to get the same running totals

**Check your understanding:** in the output above, why is the first row of `a @ U` equal to `[1, 3, 6]`?

### Exercise 4.1: the random matrix of log-increments
Build the `N × K` matrix `X`. Notice that `drift_term` and `shock_term` from Level 1 work on a whole matrix `Z` with no changes. Remember: **rows are paths, columns are time steps.**

In [ ]:
def increment_matrix(mu: float, sigma: float, t: float,
                     n_steps: int, n_paths: int) -> np.ndarray:
    """Return an n_paths x n_steps matrix whose entry [i][j] is the log-increment
    (drift + shock) of path i at time step j.
    """
    dt = ___ / n_steps
    z = np.random.normal(0, 1, size=(___, ___))
    return drift_term(mu, sigma, dt) + ___(sigma, dt, z)

In [ ]:
# Check 4.1
np.random.seed(0)
X = increment_matrix(0.10, 0.30, 1.0, 100, 2000)
assert X.shape == (2000, 100)
assert abs(X.mean() - drift_term(0.10, 0.30, 0.01)) < 5e-4   # average increment ~ drift
assert abs(X.std() - 0.03) < 0.002                           # spread ~ sigma * sqrt(dt)
print("Exercise 4.1 passed ✅")

### Exercise 4.2: the running-total matrix
Build the `K × K` upper-triangular matrix of ones. For `K = 3` it is `[[1, 1, 1], [0, 1, 1], [0, 0, 1]]`.

In [ ]:
def running_total_matrix(n_steps: int) -> np.ndarray:
    """Return the n_steps x n_steps upper-triangular matrix of ones, so that
    multiplying a row vector by it gives that row's running totals.

    >>> running_total_matrix(3).tolist()
    [[1.0, 1.0, 1.0], [0.0, 1.0, 1.0], [0.0, 0.0, 1.0]]
    """
    return np.___(np.ones((___, ___)))

In [ ]:
# Check 4.2
U = running_total_matrix(3)
assert U.shape == (3, 3)
assert U.tolist() == [[1, 1, 1], [0, 1, 1], [0, 0, 1]]
assert (np.array([[1.0, 2.0, 3.0]]) @ U).tolist() == [[1.0, 3.0, 6.0]]
print("Exercise 4.2 passed ✅")

### Exercise 4.3: all paths at once
Combine the pieces. The result has shape `n_paths × (n_steps + 1)`: one row per path, with the starting price in column 0 (the same layout as the list of lists from Level 3). Use `@` for the matrix product. A plain `*` multiplies entry by entry, which is **not** the same thing.

In [ ]:
def simulate_paths_vectorized(s0: float, mu: float, sigma: float, t: float,
                               n_steps: int, n_paths: int) -> np.ndarray:
    """Return an n_paths x (n_steps + 1) array of GBM prices; row i is path i."""
    x = increment_matrix(mu, sigma, t, n_steps, n_paths)
    u = running_total_matrix(n_steps)
    log_changes = x ___ u                                         # running totals
    log_changes = np.hstack([np.zeros((n_paths, 1)), log_changes])  # add the starting column
    return s0 * np.___(log_changes)                                # back to prices

In [ ]:
# Check 4.3
np.random.seed(1)
P = simulate_paths_vectorized(S0, MU, SIGMA, T, N_STEPS, 2000)
assert P.shape == (2000, N_STEPS + 1)
assert (P[:, 0] == S0).all()
assert (P > 0).all()
assert abs(P[:, -1].mean() - S0 * math.exp(MU * T)) < 4     # same sanity check as Exercise 3.3
print("Exercise 4.3 passed ✅")

### Exercise 4.4: the built-in shortcut
Building a `K × K` matrix just to add things up wastes memory when `K` is large. NumPy has a built-in for running totals: `np.cumsum(x, axis=1)`. Write the same function using it. **No blanks this time.** Your answer should match Exercise 4.3 exactly when both use the same random numbers.

In [ ]:
def simulate_paths_cumsum(s0: float, mu: float, sigma: float, t: float,
                          n_steps: int, n_paths: int) -> np.ndarray:
    """Same result as simulate_paths_vectorized, but use np.cumsum(..., axis=1)
    instead of matrix multiplication.
    """
    # TODO: write the function body
    pass

In [ ]:
# Check 4.4
np.random.seed(11)
p1 = simulate_paths_vectorized(100.0, 0.10, 0.30, 1.0, 50, 20)
np.random.seed(11)
p2 = simulate_paths_cumsum(100.0, 0.10, 0.30, 1.0, 50, 20)
assert p2.shape == (20, 51)
assert np.allclose(p1, p2)
print("Exercise 4.4 passed ✅")

### Exercise 4.5: loop vs. vectorized: how much faster?
This cell is complete. Run it and compare the times. (Exact numbers depend on your computer.)

In [ ]:
import time

big_paths, big_steps = 5000, 252

start = time.perf_counter()
loop_result = simulate_paths(S0, MU, SIGMA, T, big_steps, big_paths)
loop_time = time.perf_counter() - start

start = time.perf_counter()
vec_result = simulate_paths_cumsum(S0, MU, SIGMA, T, big_steps, big_paths)
vec_time = time.perf_counter() - start

print(f"Loop version:       {loop_time:.3f} s,  mean final price {average(final_prices(loop_result)):.2f}")
print(f"Vectorized version: {vec_time:.3f} s,  mean final price {vec_result[:, -1].mean():.2f}")
print(f"Theory:                      mean final price {S0 * math.exp(MU * T):.2f}")

### Exercise 4.6: visualize the vectorized paths
`plt.plot(x, y)` draws one line per **column** of `y`, but our rows are paths. Transposing a NumPy array with `.T` swaps rows and columns. Fill in the blanks.

In [ ]:
np.random.seed(4)
paths_np = simulate_paths_cumsum(S0, MU, SIGMA, T, N_STEPS, M)
times = make_times(T, N_STEPS)

plt.figure(figsize=(8, 4))
plt.plot(times, paths_np[:50].___, linewidth=0.7, alpha=0.6)
plt.xlabel("Years (t)")
plt.ylabel("Stock Price (S_t)")
plt.title(f"Vectorized GBM: S0={___}, mu={___}, sigma={___}")
plt.show()

**Short answer questions** (double-click to edit):

1. In the random matrix `Z`, what does one row represent? What does one column represent?
   *Your answer:*
2. Work out by hand: what is `[2, 1, 4] @ U` for the 3 × 3 matrix `U` from the primer? Why does multiplying by `U` produce running totals?
   *Your answer:*
3. Why do we take a running total of the **log**-increments and apply `exp` at the end, instead of adding up prices directly?
   *Your answer:*
4. Why is the vectorized version so much faster than the loop? (Hint: where does the looping happen?)
   *Your answer:*

---
## ⭐⭐⭐⭐ Level 5 – Challenge: estimate the parameters from data

Real analysts work backwards: given a history of prices, estimate `mu` and `sigma`. The key idea is that the **log returns** `ln(P[i+1] / P[i])` of a GBM are normally distributed with:
- mean = `(mu - 0.5*sigma**2) * dt`
- standard deviation = `sigma * sqrt(dt)`

So `sigma_hat = std(log_returns) / sqrt(dt)` and `mu_hat = mean(log_returns) / dt + 0.5 * sigma_hat**2`.

### Exercise 5.1: log returns and standard deviation
Write both functions. Use `math.log`. For the standard deviation use the **sample** formula (divide by `n - 1`).

In [ ]:
def log_returns(prices: List[float]) -> List[float]:
    """Return the list of log returns ln(prices[i + 1] / prices[i]).
    The result has len(prices) - 1 entries.

    >>> [round(r, 4) for r in log_returns([100.0, 110.0, 99.0])]
    [0.0953, -0.1054]
    """
    # TODO: write the function body
    pass


def std_dev(values: List[float]) -> float:
    """Return the sample standard deviation (divide by n - 1) of a list
    with at least two numbers.

    >>> round(std_dev([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]), 4)
    2.1381
    """
    # TODO: write the function body
    pass

In [ ]:
# Check 5.1
r = log_returns([100.0, 110.0, 99.0])
assert len(r) == 2
assert abs(r[0] - math.log(1.1)) < 1e-12
assert abs(std_dev([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]) - 2.13809) < 1e-4
print("Exercise 5.1 passed ✅")

### Exercise 5.2: estimate `mu` and `sigma`
Return a tuple `(mu_hat, sigma_hat)`.

In [ ]:
def estimate_mu_sigma(prices: List[float], dt: float) -> Tuple[float, float]:
    """Return (mu_hat, sigma_hat) estimated from a price history whose
    consecutive prices are dt years apart.
    """
    r = ___(prices)
    sigma_hat = ___(r) / math.sqrt(dt)
    mu_hat = ___(r) / dt + 0.5 * sigma_hat ** 2
    return (mu_hat, sigma_hat)

In [ ]:
# Check 5.2: simulate a LONG history, then see if we recover the true parameters
random.seed(7)
long_path = simulate_path(100.0, 0.08, 0.20, 100.0, 25200)   # 100 years of daily data
mu_hat, sigma_hat = estimate_mu_sigma(long_path, 100.0 / 25200)
print(f"mu_hat = {mu_hat:.3f} (true 0.08),  sigma_hat = {sigma_hat:.3f} (true 0.20)")
assert abs(sigma_hat - 0.20) < 0.02
assert abs(mu_hat - 0.08) < 0.10
print("Exercise 5.2 passed ✅")

**Reflection:** `sigma_hat` is very accurate but `mu_hat` is noisy, even with 100 years of data. Try 1 year of data (`T = 1`, 252 steps) a few times. Why is it so much harder to estimate the expected return than the volatility? (Write 2 to 3 sentences.)

*Your answer:*

---
## ⚠️ Common Mistakes (read before you debug!)

| # | Mistake | Symptom | Fix |
|---|---|---|---|
| 1 | **Forgetting the `- 0.5 * sigma**2` correction** | Simulated average drifts above the 110.52 target | Use `(mu - 0.5 * sigma ** 2) * dt` |
| 2 | **Using `T` instead of `dt = T / n_steps` inside the loop** | Prices explode or collapse (huge numbers) | Compute `dt` once, before the loop |
| 3 | **Integer division confusion**, e.g. writing `T // n_steps` | `dt` becomes `0` and the price never moves | Use `/`, not `//` |
| 4 | **`sqrt(dt)` vs `dt`** in the shock term | Volatility looks far too small | Shock uses `math.sqrt(dt)`; drift uses `dt` |
| 5 | **Off-by-one path length**, e.g. `range(n_steps + 1)` | Path has `n_steps + 2` entries | Path = start price + `n_steps` new prices = `n_steps + 1` |
| 6 | **Drawing `z` once outside the loop** | Path is a perfectly smooth curve | Call `random.gauss(0, 1)` *inside* the loop, each step |
| 7 | **Reusing the same list**, `[path] * n_paths` | All "different" paths are identical | Build a fresh list each iteration |
| 8 | **Using `path.append(x)` result**: `path = path.append(x)` | `TypeError: 'NoneType' has no len()` | `append` changes the list in place and returns `None` |
| 9 | **Percent vs decimal**: `mu = 8` instead of `0.08` | Prices become astronomically large | Rates are decimals: 8% = `0.08` |
| 10 | **`math.log` vs `math.exp` mix-up** | Prices near zero or wildly wrong | `exp` builds prices; `log` extracts returns |
| 11 | **Mixing units**: `mu` per year but `dt` in days | Drift and volatility off by a large factor (e.g. 252) | Keep everything in years: one trading day is about `1/252` |
| 12 | **Forgetting `return`** | Function gives back `None` | Every function with a return type needs `return` |
| 13 | **Plotting against step number instead of years** | x-axis runs 0 to 100 instead of 0 to 1 | Build `times = make_times(T, N_STEPS)` and call `plt.plot(times, path)` |
| 14 | **Adding log prices but forgetting `math.exp`** | "Prices" stay near 4.6 (which is `log(100)`) | Convert back with `math.exp` before appending to the path |
| 15 | **Wrong `axis` in `np.cumsum` / `sum`** | Each path ends up adding up *other paths* instead of its own history | Rows are paths and columns are time, so accumulate over time with `axis=1` |
| 16 | **Swapping the matrix shape**: `size=(n_steps, n_paths)` | Result has shape `(n_steps + 1, n_paths)`; checks and plots look transposed | `size=(n_paths, n_steps)`: one row per path |
| 17 | **Using `*` instead of `@`** | Wrong numbers or a shape error | `*` is entry-by-entry; `@` is matrix multiplication |
| 18 | **Forgetting the starting column** | Array has `n_steps` columns and `[:, 0]` is not `S0` | `np.hstack` a column of zeros (log of 1) before applying `exp` |
| 19 | **`math.exp` on an array** | `TypeError: only length-1 arrays can be converted` | Use `np.exp` for arrays; `math.exp` only for single numbers |
| 20 | **Plotting rows as if they were columns** | One line instead of many, or a shape error | Transpose with `.T` so each path is a column |

---
## 🔑 Takeaways

**Finance / math ideas**
- **GBM splits each move into drift and shock.** Drift `(mu - 0.5*sigma**2) * dt` is the predictable part; shock `sigma * sqrt(dt) * Z` is the random part.
- **Multiplying by `exp(...)` keeps prices positive** and makes returns, not price changes, the thing that is random.
- **The `-0.5*sigma**2` term is a correction, not a typo.** Without it the simulated average price drifts above `S0 * exp(mu * T)`.
- **Volatility scales with `sqrt(dt)`, drift scales with `dt`.** This is why units (years, not days) matter.
- **Monte Carlo means running many random trials and averaging.** More paths give an answer closer to theory, but never exactly equal to it.
- **GBM has an explicit solution:** the log price is normal, so prices are log-normal. That is why multiplying step factors and adding log returns (then `exp`) give the same path.
- **Volatility is easier to estimate than expected return.** `sigma_hat` was accurate; `mu_hat` stayed noisy even with 100 years of data.

**Programming skills you practiced**
- **Break a big problem into small functions** (`drift_term` → `gbm_step` → `simulate_path` → `simulate_paths`), where each function reuses the one before it.
- **Build lists with an accumulator loop:** start with `[s0]` or `[]`, then `append` inside a `for` loop.
- **Read a docstring as a contract:** the type annotations and examples tell you exactly what to return.
- **Turn step numbers into meaningful units** (years) before plotting, and label axes and titles so a chart can stand on its own.
- **Test against a known answer.** Comparing the simulation to `S0 * exp(mu * T)` is how you know your code is right, not just that it runs.
- **Off-by-one and unit errors are the most common bugs.** Check list lengths and keep `dt` in years.

- **Vectorize:** generate all the randomness in one `n_paths × n_steps` matrix, then turn it into prices with whole-array operations. Rows are paths, columns are time.
- **A running total is a matrix product:** multiplying by an upper-triangular matrix of ones does what `np.cumsum` does; the built-in is the efficient version.
- **Check shapes first:** most NumPy bugs are a swapped `axis` or a transposed matrix.

**Self-check:** you should be able to explain, in your own words, (1) what `dt` is, (2) why `z` is drawn inside the loop, and (3) why a path has `n_steps + 1` prices, (4) the two ways to build a path (multiply factors vs. add log returns), and (5) what the rows and columns of the random matrix mean in the vectorized version.

---
## 🎓 Wrap-up: put it on your resume

You just built a Monte Carlo price simulator from scratch. Here are resume bullets you can adapt (edit them to reflect what *you* did):

- *Implemented a Geometric Brownian Motion simulator in Python to generate thousands of stock price paths; validated the model against the analytical expected value E[S_T] = S0·e^(μT).*
- *Estimated drift and volatility from simulated price histories using log returns and sample standard deviation, and analyzed why volatility is estimated more accurately than expected return.*
- *Vectorized the Monte Carlo simulation with NumPy (matrix of normal shocks, cumulative sums of log-increments) and benchmarked it against a loop-based implementation.*
- *Wrote docstring-driven, unit-tested functions using only core Python (math, random) and visualized simulation results with matplotlib.*

**Optional extensions** (great for a GitHub project): plot a histogram of `final_prices`, estimate the probability the price ends below `S0`, add a second stock, or download real price data and run `estimate_mu_sigma` on it.

### Final step
Run `Kernel → Restart & Run All`. Every check cell should print ✅ with no errors, then submit your `.ipynb` file.